# sEEG Multi-Class Riemannian Geometry Analysis

This notebook implements a 4-class Riemannian Geometry pipeline using pyRiemann:
1. Load all trials from 4 class folders in `dataset/`
2. Detect and remove bad channels (concatenated across all trials)
3. Apply notch + bandpass filtering and compute LMP
4. Build overlapping windows with class labels
5. Extract Riemannian features using MDM (Minimum Distance to Mean)
6. Evaluate with cross-validation and visualize results

## Step 0 - Imports and Function Definitions
Define all preprocessing, CSP, and visualization functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch
from pathlib import Path
from pyriemann.estimation import Covariances
from pyriemann.classification import MDM
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


def detect_bad_channels_stat(data, standard_range=(-500, 500)):
    """Detect bad sEEG channels using statistical features.

    Args:
        data: Array with shape (n_channels, n_samples).
        standard_range: Acceptable amplitude range in microvolts.
    """
    bad_channels_indices = []

    constant_mask = np.all(np.isclose(data, data[:, [0]], atol=1e-5), axis=1)
    constant_channels = np.where(constant_mask)[0]
    bad_channels_indices.extend(constant_channels)

    means = np.mean(data, axis=1)
    offset_mask = np.abs(means) > 20
    offset_channels = np.where(offset_mask)[0]
    bad_channels_indices.extend(offset_channels)

    outside_mask = np.any(
        (data < standard_range[0]) | (data > standard_range[1]),
        axis=1,
    )
    outside_channels = np.where(outside_mask)[0]
    bad_channels_indices.extend(outside_channels)

    unique_bad_channels = sorted(set(bad_channels_indices))

    print(f"Constant channels: {constant_channels}")
    print(f"Offset channels: {offset_channels}")
    print(f"Outside standard range: {outside_channels}")

    return unique_bad_channels


def bandpass_filter(data, low_freq, high_freq, fs=2000, order=4):
    """Apply a band-pass filter to extract a target frequency band.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        low_freq: Low cutoff frequency in Hz.
        high_freq: High cutoff frequency in Hz.
        fs: Sampling rate in Hz (default: 2000).
        order: Filter order (default: 4).

    Returns:
        Filtered data with the same shape as input.
    """
    nyquist = fs / 2
    low = low_freq / nyquist
    high = high_freq / nyquist

    b, a = butter(order, [low, high], btype="band")
    filtered_data = filtfilt(b, a, data, axis=1)

    return filtered_data


def apply_notch_filter(data, freq=50, fs=2000, q=30):
    """Apply a notch filter to remove power-line interference.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        freq: Frequency to remove in Hz (50 or 60 typically).
        fs: Sampling rate in Hz.
        q: Quality factor; higher values produce a narrower notch.

    Returns:
        Filtered data with the same shape as input.
    """
    nyquist = fs / 2
    w0 = freq / nyquist

    b, a = iirnotch(w0, q)
    filtered_data = filtfilt(b, a, data, axis=1)

    return filtered_data


def create_windows(data, window_size, stride):
    """Create overlapping windows from channel-by-time data.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        window_size: Window length in samples.
        stride: Step between consecutive windows in samples.

    Returns:
        3D array with shape (n_windows, n_channels, window_size).
    """
    _, n_samples = data.shape
    windows = []
    for i in range(0, n_samples - window_size + 1, stride):
        windows.append(data[:, i : i + window_size])
    return np.array(windows)


def calculate_lmp(data, window_ms=250, fs=2000):
    """Calculate Local Motor Potential (LMP) as a running average.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        window_ms: Window size for running average in milliseconds.
        fs: Sampling rate in Hz.

    Returns:
        LMP data with the same shape as input.
    """
    window_size = int(window_ms * fs / 1000)
    window = np.ones(window_size) / window_size
    lmp_data = np.apply_along_axis(
        lambda x: np.convolve(x, window, mode="same"),
        axis=1,
        arr=data,
    )
    return lmp_data


def filter_seeg_data(data, fs=2000):
    """Run the full sEEG filtering pipeline.

    Steps:
        1) Apply notch filtering to remove power-line noise.
        2) Apply band-pass filtering to obtain low/high frequency bands.
        3) Calculate Local Motor Potential (LMP) as running average.
    """
    data_no_notch = apply_notch_filter(data, freq=50, fs=fs)
    lfb_result = bandpass_filter(data_no_notch, 0.5, 30, fs)
    hfb_result = bandpass_filter(data_no_notch, 80, 200, fs)
    lmp_result = calculate_lmp(data_no_notch, window_ms=250, fs=fs)
    return lfb_result, hfb_result, lmp_result


def extract_riemannian_features(hfb_windows, lmp_windows, labels, n_classes=4):
    """Extract Riemannian features using Tangent Space Mapping + Logistic Regression.
    
    Returns dict with trained models for each band and a fused model.
    """
    bands = {"HFB": hfb_windows, "LMP": lmp_windows}
    models = {}
    
    # Train separate models for each band
    for band_name, band_windows in bands.items():
        clf = make_pipeline(
            Covariances(estimator='lwf'),
            TangentSpace(metric='riemann'),
            LogisticRegression(penalty='l2', C=1.0, max_iter=2000, multi_class='multinomial')
        )
        clf.fit(band_windows, labels)
        models[band_name] = clf
        
    # Also train a fused feature model
    cov_est = Covariances(estimator='lwf')
    ts_est = TangentSpace(metric='riemann')
    
    # Project both bands to Tangent Space and concatenate
    hfb_ts = ts_est.fit_transform(cov_est.fit_transform(hfb_windows))
    lmp_ts = ts_est.fit_transform(cov_est.fit_transform(lmp_windows))
    fused_features = np.hstack((hfb_ts, lmp_ts))
    
    fused_clf = LogisticRegression(penalty='l2', C=1.0, max_iter=2000, multi_class='multinomial')
    fused_clf.fit(fused_features, labels)
    models['Fused'] = {
        'clf': fused_clf,
        'ts_est': ts_est,
        'cov_est': cov_est
    }
    
    return {'models': models, 'band_names': list(bands.keys()) + ['Fused']}


np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (12, 4)

## Step 1 - Load All Class Data
Load `.npz` files from each class folder. Each file is one trial with shape `(channels, samples)`.

In [2]:
dataset_dir = Path('dataset')

class_configs = [
    ('class_1_关一下灯', '关一下灯'),
    ('class_2_我冷了', '我冷了'),
    ('class_3_我想出门走走', '我想出门走走'),
    ('class_4_现在几点了', '现在几点了'),
]

class_names = [cfg[1] for cfg in class_configs]
all_trials = []

for cls_idx, (folder_name, display_name) in enumerate(class_configs):
    folder = dataset_dir / folder_name
    npz_files = sorted(folder.glob('*.npz'))
    trials = []
    for f in npz_files:
        data = np.load(f)
        arr = data[data.files[0]]
        if arr.shape[0] > arr.shape[1]:
            arr = arr.T
        trials.append(arr)
    all_trials.append(trials)
    print(f'Class {cls_idx} ({display_name}): {len(trials)} trials, shape {trials[0].shape if trials else "N/A"}')

n_classes = len(all_trials)
n_channels = all_trials[0][0].shape[0]
print(f'\nTotal: {n_classes} classes, {n_channels} channels, {sum(len(t) for t in all_trials)} trials')

Class 0 (关一下灯): 20 trials, shape (276, 8480)
Class 1 (我冷了): 20 trials, shape (276, 8420)
Class 2 (我想出门走走): 20 trials, shape (276, 8400)
Class 3 (现在几点了): 20 trials, shape (276, 8400)

Total: 4 classes, 276 channels, 80 trials


## Step 2 - Detect and Remove Bad Channels
Concatenate all trials along the time axis to get a robust estimate of bad channels, then remove them from every trial.

In [3]:
all_trials_flat = [trial for cls_trials in all_trials for trial in cls_trials]
concatenated = np.concatenate(all_trials_flat, axis=1)
print(f'Concatenated data shape: {concatenated.shape}')

bad_indices = detect_bad_channels_stat(concatenated)
print(f'\nBad channels ({len(bad_indices)} total): {bad_indices}')

keep_mask = np.ones(n_channels, dtype=bool)
keep_mask[bad_indices] = False

for cls_idx in range(n_classes):
    for trial_idx in range(len(all_trials[cls_idx])):
        all_trials[cls_idx][trial_idx] = all_trials[cls_idx][trial_idx][keep_mask]

n_channels_clean = all_trials[0][0].shape[0]
print(f'\nChannels after removal: {n_channels_clean} (removed {len(bad_indices)})')

Concatenated data shape: (276, 674560)


c:\Users\hilin\miniconda3\envs\data_env\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


Constant channels: [272 273 274 275]
Offset channels: [256 257 258 259 260 261 262 263 264 265 266 267 272 273 274]
Outside standard range: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  24
  25  26  27  30  31  35  39  40  41  42  43  44  45  46  47  49  50  51
  52  53  54  55  56  57  58  59  60  61  62  63  64  65  73  74  75  77
  78  79  80  81  82  85  94  95  96  97  98  99 100 101 102 103 104 105
 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123
 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141
 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159
 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177
 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195
 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213
 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231
 232 233 234 235 236 237 238 239 240 241 242 243 244 245 

## Step 3 - Filter All Trials
Apply the full preprocessing pipeline (notch -> HFB/LFB -> LMP) to every trial.

In [4]:
fs = 2000

all_hfb_trials = []
all_lmp_trials = []

for cls_idx in range(n_classes):
    cls_hfb = []
    cls_lmp = []
    for trial in all_trials[cls_idx]:
        _, hfb, lmp = filter_seeg_data(trial, fs=fs)
        cls_hfb.append(hfb)
        cls_lmp.append(lmp)
    all_hfb_trials.append(cls_hfb)
    all_lmp_trials.append(cls_lmp)

print(f'HFB trial shapes: {[t.shape for t in all_hfb_trials[0][:3]]}...')
print(f'LMP trial shapes: {[t.shape for t in all_lmp_trials[0][:3]]}...')

HFB trial shapes: [(38, 8480), (38, 8400), (38, 8400)]...
LMP trial shapes: [(38, 8480), (38, 8400), (38, 8400)]...


## Step 4 - Create Overlapping Windows with Labels
Cut each trial into overlapping windows and assign the trial's class label to all its windows.

In [5]:
window_size = 250
stride = 225

hfb_windows_list = []
lmp_windows_list = []
labels_list = []

for cls_idx in range(n_classes):
    for trial_hfb, trial_lmp in zip(all_hfb_trials[cls_idx], all_lmp_trials[cls_idx]):
        hfb_w = create_windows(trial_hfb, window_size=window_size, stride=stride)
        lmp_w = create_windows(trial_lmp, window_size=window_size, stride=stride)
        n_w = len(hfb_w)
        hfb_windows_list.append(hfb_w)
        lmp_windows_list.append(lmp_w)
        labels_list.append(np.full(n_w, cls_idx, dtype=int))

hfb_windows = np.concatenate(hfb_windows_list, axis=0)
lmp_windows = np.concatenate(lmp_windows_list, axis=0)
labels = np.concatenate(labels_list, axis=0)

print(f'Total windows: {len(labels)}')
print(f'HFB windows shape: {hfb_windows.shape}')
print(f'LMP windows shape: {lmp_windows.shape}')
print(f'\nWindows per class:')
for cls_idx, name in enumerate(class_names):
    count = np.sum(labels == cls_idx)
    print(f'  Class {cls_idx} ({name}): {count} windows')

Total windows: 2960
HFB windows shape: (2960, 38, 250)
LMP windows shape: (2960, 38, 250)

Windows per class:
  Class 0 (关一下灯): 740 windows
  Class 1 (我冷了): 740 windows
  Class 2 (我想出门走走): 740 windows
  Class 3 (现在几点了): 740 windows


## Step 5 - Multi-Class Riemannian Tangent Space classification
For each band (HFB and LMP), extract covariance matrices, project to a Tangent Space, and classify with LogisticRegression. Additionally, fuse the Tangent Space features.

In [ ]:
riemannian_result = extract_riemannian_features(
    hfb_windows, lmp_windows, labels,
    n_classes=n_classes
)

print(f'Riemannian models trained for bands: {riemannian_result["band_names"]}')
for band_name in riemannian_result["band_names"]:
    print(f'  {band_name}: trained Tangent Space classifier')

Riemannian models trained for bands: ['HFB', 'LMP']
  HFB: trained MDM classifier
  LMP: trained MDM classifier


## Step 6 - Compute Covariances
Compute the covariance matrices for the HFB and LMP bands using the LwF estimator.

In [7]:
# Compute covariance matrices for visualization and inspection (optional)

cov_est = Covariances(estimator='lwf')
hfb_cov = cov_est.fit_transform(hfb_windows)
lmp_cov = cov_est.fit_transform(lmp_windows)

print(f'HFB covariance shape: {hfb_cov.shape}')
print(f'LMP covariance shape: {lmp_cov.shape}')

HFB covariance shape: (2960, 38, 38)
LMP covariance shape: (2960, 38, 38)


## Step 7 - Prediction and Evaluation
Since the Riemannian MDM approach doesn't require traditional feature selection, we can directly use the trained models for prediction and evaluation.

In [ ]:
def predict_riemannian(riemannian_result, hfb_windows, lmp_windows, fusion='fused_features'):
    """Predict using trained TSM models with fusion strategy.
    
    Args:
        fusion: 'majority' (vote from both bands), 
                'average' (mean distances/probas), or 
                'fused_features' (use the combined model)
    """
    if fusion == 'fused_features':
        # Feature fusion model
        ts_est = riemannian_result["models"]["Fused"]['ts_est']
        cov_est = riemannian_result["models"]["Fused"]['cov_est']
        opt_clf = riemannian_result["models"]["Fused"]['clf']
        
        hfb_ts = ts_est.transform(cov_est.transform(hfb_windows))
        lmp_ts = ts_est.transform(cov_est.transform(lmp_windows))
        fused_features = np.hstack((hfb_ts, lmp_ts))
        return opt_clf.predict(fused_features)
        
    hfb_preds = riemannian_result["models"]["HFB"].predict(hfb_windows)
    lmp_preds = riemannian_result["models"]["LMP"].predict(lmp_windows)
    
    if fusion == 'majority':
        return np.array([np.bincount([h, l]).argmax() for h, l in zip(hfb_preds, lmp_preds)])
    else:
        hfb_dist = riemannian_result["models"]["HFB"].predict_proba(hfb_windows)
        lmp_dist = riemannian_result["models"]["LMP"].predict_proba(lmp_windows)
        avg_dist = (hfb_dist + lmp_dist) / 2
        return np.argmax(avg_dist, axis=1)

# Test prediction with Fused Tangent Space Classification
predictions = predict_riemannian(riemannian_result, hfb_windows, lmp_windows, fusion='fused_features')
accuracy = np.mean(predictions == labels)
print(f'Overall accuracy (Feature Fusion TSM): {accuracy:.3f}')
print(f'\nClassification Report:')
print(classification_report(labels, predictions, target_names=class_names))

Overall accuracy (majority voting): 0.453

Classification Report:
              precision    recall  f1-score   support

        关一下灯       0.45      0.69      0.55       740
         我冷了       0.45      0.59      0.51       740
      我想出门走走       0.41      0.41      0.41       740
       现在几点了       0.85      0.12      0.21       740

    accuracy                           0.45      2960
   macro avg       0.54      0.45      0.42      2960
weighted avg       0.54      0.45      0.42      2960



## Step 8 - Comparison and Analysis
Compare the Riemannian MDM approach with traditional classifiers (optional).
Note: The Riemannian approach is already a complete classification pipeline and doesn't require additional classifiers.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print('=== Riemannian Tangent Space Mapping Results ===')

# Compute CV scores for each band
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = {}
for band_name, band_data in [("HFB", hfb_windows), ("LMP", lmp_windows)]:
    clf = riemannian_result["models"][band_name]
    band_scores = cross_val_score(clf, band_data, labels, cv=cv, scoring='accuracy')
    cv_scores[band_name] = {'mean': band_scores.mean(), 'std': band_scores.std()}
    print(f'{band_name} Band CV Accuracy: {cv_scores[band_name]["mean"]:.3f} ± {cv_scores[band_name]["std"]:.3f}')

# Fused
ts_est = riemannian_result["models"]["Fused"]['ts_est']
cov_est = riemannian_result["models"]["Fused"]['cov_est']
opt_clf = riemannian_result["models"]["Fused"]['clf']

hfb_ts = ts_est.transform(cov_est.transform(hfb_windows))
lmp_ts = ts_est.transform(cov_est.transform(lmp_windows))
fused_features = np.hstack((hfb_ts, lmp_ts))

fused_scores = cross_val_score(opt_clf, fused_features, labels, cv=cv, scoring='accuracy')
print(f'Feature Fusion CV Accuracy: {fused_scores.mean():.3f} ± {fused_scores.std():.3f}')

print('\n=== Summary ===')
print('The Riemannian TSM + LogReg pipeline provides:')
print('  1. High dimensionality (38 -> ~741 features) managed by L2 Regularization.')
print('  2. No need for manual or heuristic feature selection (CSP spatial filters).')
print('  3. Tangent Space Mapping unlocks traditional Euclidean classifiers while preserving topology.')
print('  4. Multi-band concatenation naturally learns to weight HFB vs LMP importance in the projection.')

=== Riemannian MDM Results ===
HFB Band CV Accuracy: 0.320 ± 0.010
LMP Band CV Accuracy: 0.297 ± 0.016

=== Summary ===
The Riemannian MDM approach provides:
  1. Direct classification without feature selection
  2. Robustness to electrode impedance changes
  3. Natural multi-class support
  4. Interpretable results (distance to class means)
